# Layer 2 — Behavioral Feature Engineering
## MerRec C2C Interaction Dataset

**Capstone Project** | American University of Armenia  
**Student:** Arevik Melikyan  
**Supervisor:** Varazdat Stepanyan

---

### Purpose

Layer 1 characterised the raw data. Layer 2 transforms it into three structured feature tables that serve as the inputs to all modelling in Layer 3 and the interpretability vocabulary for Layer 5.

Every feature is computed entirely in DuckDB SQL — no row-by-row Python loops — so the full ~1B row dataset is processed without loading it into RAM.

### Outputs

| Table | Granularity | Saved to |
|-------|-------------|---------|
| `user_features.parquet` | One row per user | `DATASET_ROOT/features/` |
| `item_features.parquet` | One row per item | `DATASET_ROOT/features/` |
| `session_features.parquet` | One row per session | `DATASET_ROOT/features/` |

### Design decisions

- **Cold-start threshold `k = 20`** — items with ≤ 20 interactions receive a cold-start flag and will rely on category-level priors in Layer 3.
- **Price tiers** — price is bucketed into 5 quantile bands (Q1–Q5) plus retained as `log_price` for continuous models.
- **Session ID** — `session_id` is present and reliable in the raw data; no reconstruction needed.
- **Event weights** — high-intent events (`item_add_to_cart_tap`, `buy_comp`) are tracked separately from passive views to distinguish intent levels.
- **All features are computed as of the dataset's latest timestamp** — recency is relative to `MAX(stime)`, not wall-clock time, ensuring reproducibility.

### Sections
1. Environment setup  
2. Schema validation  
3. User feature table  
4. Item feature table  
5. Session feature table  
6. Feature validation & quality checks  
7. Cross-table join sample  


---
## 1. Environment Setup

Same DuckDB connection pattern as Layer 1. Output directory `DATASET_ROOT/features/` is created if it does not exist.


In [ ]:
import os
import warnings
from pathlib import Path

import duckdb
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns

warnings.filterwarnings("ignore")

RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)

# ── Paths ─────────────────────────────────────────────────────────────────────
DATASET_ROOT  = Path(os.environ.get("DATASET_ROOT", "/Volumes/T5 EVO"))
DATASET_PATH  = DATASET_ROOT / "hf" / "merrec"
FEATURES_DIR  = DATASET_ROOT / "features"
FEATURES_DIR.mkdir(parents=True, exist_ok=True)

USER_FEATURES_PATH    = str(FEATURES_DIR / "user_features.parquet")
ITEM_FEATURES_PATH    = str(FEATURES_DIR / "item_features.parquet")
SESSION_FEATURES_PATH = str(FEATURES_DIR / "session_features.parquet")

# ── DuckDB ────────────────────────────────────────────────────────────────────
con = duckdb.connect(database=":memory:")
con.execute("PRAGMA threads=10")
con.execute("PRAGMA memory_limit='20GB'")

con.execute(f"""
CREATE VIEW merrec AS
SELECT *
FROM read_parquet('{str(DATASET_PATH)}/**/*.parquet')
""")

print("View created.")
print(f"Features will be saved to: {str(FEATURES_DIR)}")


---
## 2. Schema Validation

Before computing any features, we confirm every required column is present and discover all `event_id` values in the dataset. Since Layer 1 has not been run yet, this section also serves as a first look at the data.


In [ ]:
# ── 2a. Confirm required columns ──────────────────────────────────────────────
schema = con.execute("DESCRIBE merrec").df()
print("Full schema:")
display(schema)

REQUIRED_COLUMNS = {
    "user_id", "item_id", "session_id",
    "stime", "event_id", "price",
    "c0_name", "c1_name", "c2_name"
}

present = set(schema["column_name"].str.strip().tolist())
missing = REQUIRED_COLUMNS - present

if missing:
    raise ValueError(f"MISSING COLUMNS: {missing}. Check dataset path and parquet schema.")
else:
    print(f"\n  All {len(REQUIRED_COLUMNS)} required columns confirmed present.")


In [ ]:
# ── 2b. Dataset scale ─────────────────────────────────────────────────────────
scale = con.execute("""
SELECT
    COUNT(*)                            AS total_events,
    APPROX_COUNT_DISTINCT(user_id)      AS approx_users,
    APPROX_COUNT_DISTINCT(item_id)      AS approx_items,
    APPROX_COUNT_DISTINCT(session_id)   AS approx_sessions,
    COUNT(DISTINCT event_id)            AS event_types,
    MIN(stime)                          AS earliest,
    MAX(stime)                          AS latest
FROM merrec
""").df()
display(scale.T.rename(columns={0: "value"}))


In [ ]:
# ── 2c. Discover all event_id values ──────────────────────────────────────────
event_types = con.execute("""
SELECT event_id, COUNT(*) AS cnt
FROM merrec
GROUP BY event_id
ORDER BY cnt DESC
""").df()

print("All event types in dataset:")
display(event_types)

ALL_EVENT_IDS = event_types["event_id"].tolist()


In [ ]:
# ── 2d. Set event name constants — adjust if your dataset uses different names ──
VIEW_EVENT     = "item_view"
LIKE_EVENT     = "item_like"
CART_EVENT     = "item_add_to_cart_tap"
PURCHASE_EVENT = "buy_comp"
COLD_START_K   = 20

for name, val in [
    ("VIEW_EVENT",     VIEW_EVENT),
    ("LIKE_EVENT",     LIKE_EVENT),
    ("CART_EVENT",     CART_EVENT),
    ("PURCHASE_EVENT", PURCHASE_EVENT),
]:
    found  = val in ALL_EVENT_IDS
    status = "  found" if found else "  NOT FOUND — update constant above"
    print(f"  {name:20s} = '{val}'{status}")

print(f"\n  COLD_START_K = {COLD_START_K}")


---
## 3. User Feature Table

One row per user. Features are grouped into five conceptual families:

| Family | Features | Research question |
|--------|----------|-------------------|
| **RFM** | recency, frequency, monetary proxy | Engagement level |
| **Breadth** | category entropy, top category, category count | Specialist vs explorer |
| **Longevity** | active lifespan, first/last event | Platform loyalty |
| **Session behaviour** | session count, avg/max session length, bounce rate | Short-term pattern proxy |
| **Funnel depth** | view/like/cart/purchase counts, conversion rate | Purchase propensity |

Recency is computed relative to `MAX(stime)` in the dataset so results are reproducible.


In [ ]:
# ── 3a. RFM + funnel counts ───────────────────────────────────────────────────
print("Computing user RFM + funnel features...")

con.execute(f"""
CREATE OR REPLACE VIEW user_rfm AS
WITH ref AS (SELECT MAX(stime) AS max_ts FROM merrec)
SELECT
    user_id,
    DATEDIFF('day', MAX(stime), (SELECT max_ts FROM ref)) AS recency_days,
    COUNT(*)                                               AS frequency,
    COUNT(*) FILTER (WHERE event_id = '{PURCHASE_EVENT}') AS purchase_count,
    COUNT(*) FILTER (WHERE event_id = '{VIEW_EVENT}')     AS view_count,
    COUNT(*) FILTER (WHERE event_id = '{LIKE_EVENT}')     AS like_count,
    COUNT(*) FILTER (WHERE event_id = '{CART_EVENT}')     AS cart_count,
    CASE
        WHEN COUNT(*) FILTER (WHERE event_id = '{VIEW_EVENT}') > 0
        THEN COUNT(*) FILTER (WHERE event_id = '{PURCHASE_EVENT}') * 1.0
             / COUNT(*) FILTER (WHERE event_id = '{VIEW_EVENT}')
        ELSE 0
    END                                                    AS view_to_purchase_rate,
    MIN(stime)                                             AS first_event_ts,
    MAX(stime)                                             AS last_event_ts,
    DATEDIFF('day', MIN(stime), MAX(stime))               AS active_lifespan_days
FROM merrec
GROUP BY user_id
""")

print("  user_rfm view created")
display(con.execute("SELECT * FROM user_rfm LIMIT 5").df())


In [ ]:
# ── 3b. Category preference features ─────────────────────────────────────────
#
# Shannon entropy H of user's L0 category distribution.
# H = 0   → all interactions in one category  (specialist)
# H = max → uniform spread across categories  (explorer)

print("Computing user category preference features...")

con.execute("""
CREATE OR REPLACE VIEW user_category AS
WITH user_cat_counts AS (
    SELECT user_id, c0_name, c1_name, COUNT(*) AS cnt
    FROM merrec
    WHERE c0_name IS NOT NULL
    GROUP BY user_id, c0_name, c1_name
),
user_totals AS (
    SELECT user_id, SUM(cnt) AS total
    FROM user_cat_counts GROUP BY user_id
),
entropy_calc AS (
    SELECT
        uc.user_id,
        -SUM((uc.cnt * 1.0 / ut.total) * LOG2(uc.cnt * 1.0 / ut.total)) AS category_entropy,
        COUNT(DISTINCT uc.c0_name) AS distinct_categories_l0,
        COUNT(DISTINCT uc.c1_name) AS distinct_categories_l1
    FROM user_cat_counts uc
    JOIN user_totals ut USING(user_id)
    GROUP BY uc.user_id
),
top_cat AS (
    SELECT DISTINCT ON (user_id) user_id, c0_name AS top_category_l0
    FROM user_cat_counts
    ORDER BY user_id, cnt DESC
)
SELECT e.user_id, e.category_entropy,
       e.distinct_categories_l0, e.distinct_categories_l1,
       t.top_category_l0
FROM entropy_calc e
LEFT JOIN top_cat t USING(user_id)
""")

print("  user_category view created")
display(con.execute("SELECT * FROM user_category LIMIT 5").df())


In [ ]:
# ── 3c. Session behaviour aggregates ─────────────────────────────────────────
print("Computing user session behaviour features...")

con.execute("""
CREATE OR REPLACE VIEW user_sessions AS
WITH session_lengths AS (
    SELECT
        user_id, session_id,
        COUNT(*) AS session_len,
        DATEDIFF('second', MIN(stime), MAX(stime)) AS session_duration_sec
    FROM merrec
    GROUP BY user_id, session_id
)
SELECT
    user_id,
    COUNT(DISTINCT session_id)                                              AS session_count,
    AVG(session_len)                                                        AS avg_session_length,
    MAX(session_len)                                                        AS max_session_length,
    AVG(session_duration_sec)                                               AS avg_session_duration_sec,
    SUM(CASE WHEN session_len = 1 THEN 1 ELSE 0 END) * 1.0
        / COUNT(DISTINCT session_id)                                        AS user_bounce_rate
FROM session_lengths
GROUP BY user_id
""")

print("  user_sessions view created")
display(con.execute("SELECT * FROM user_sessions LIMIT 5").df())


In [ ]:
# ── 3d. Write final user feature table ───────────────────────────────────────
print("Joining user views and writing to Parquet...")

con.execute(f"""
COPY (
    SELECT
        r.user_id,

        -- RFM
        r.recency_days,
        r.frequency,
        r.purchase_count,
        r.view_count,
        r.like_count,
        r.cart_count,
        r.view_to_purchase_rate,
        LN(r.frequency + 1)                        AS log_frequency,

        -- Longevity
        r.first_event_ts,
        r.last_event_ts,
        r.active_lifespan_days,

        -- Category preference
        c.category_entropy,
        c.distinct_categories_l0,
        c.distinct_categories_l1,
        c.top_category_l0,

        -- Session behaviour
        s.session_count,
        s.avg_session_length,
        s.max_session_length,
        s.avg_session_duration_sec,
        s.user_bounce_rate,

        -- RFM quartile tiers (1=lowest, 4=highest)
        NTILE(4) OVER (ORDER BY r.recency_days DESC) AS recency_tier,
        NTILE(4) OVER (ORDER BY r.frequency)         AS frequency_tier,
        NTILE(4) OVER (ORDER BY r.purchase_count)    AS monetary_tier

    FROM user_rfm r
    LEFT JOIN user_category c USING(user_id)
    LEFT JOIN user_sessions  s USING(user_id)
    ORDER BY r.user_id
) TO '{USER_FEATURES_PATH}' (FORMAT PARQUET, COMPRESSION SNAPPY)
""")

n    = con.execute(f"SELECT COUNT(*) FROM read_parquet('{USER_FEATURES_PATH}')").fetchone()[0]
cols = con.execute(f"DESCRIBE SELECT * FROM read_parquet('{USER_FEATURES_PATH}')").df()
print(f"  user_features.parquet written")
print(f"  Rows: {n:,}   Columns: {len(cols)}")


---
## 4. Item Feature Table

One row per item. The `cold_start_flag` is the most consequential output: items with ≤ 20 interactions cannot form reliable latent representations from interactions alone and fall back to category-level priors in Layer 3.

| Family | Features | Purpose |
|--------|----------|---------|
| **Popularity** | total events, unique users, view/like/cart/purchase counts | Interaction volume |
| **Conversion** | view-to-purchase rate, like rate | Demand signal quality |
| **Temporal** | freshness, attention lifespan | Listing lifecycle stage |
| **Price** | log price, price tier Q1–Q5 | Price segment positioning |
| **Category** | L0 / L1 / L2 names | Cold-start prior & context |
| **Cold-start** | cold_start_flag, interaction_tier | Representation quality flag |


In [ ]:
# ── 4a. Interaction counts & conversion ───────────────────────────────────────
print("Computing item interaction + conversion features...")

con.execute(f"""
CREATE OR REPLACE VIEW item_interactions AS
SELECT
    item_id,
    COUNT(*)                                                          AS total_events,
    APPROX_COUNT_DISTINCT(user_id)                                    AS unique_users,
    COUNT(*) FILTER (WHERE event_id = '{VIEW_EVENT}')                AS view_count,
    COUNT(*) FILTER (WHERE event_id = '{LIKE_EVENT}')                AS like_count,
    COUNT(*) FILTER (WHERE event_id = '{CART_EVENT}')                AS cart_count,
    COUNT(*) FILTER (WHERE event_id = '{PURCHASE_EVENT}')            AS purchase_count,
    APPROX_COUNT_DISTINCT(session_id)                                 AS session_reach,
    CASE
        WHEN COUNT(*) FILTER (WHERE event_id = '{VIEW_EVENT}') > 0
        THEN COUNT(*) FILTER (WHERE event_id = '{PURCHASE_EVENT}') * 1.0
             / COUNT(*) FILTER (WHERE event_id = '{VIEW_EVENT}')
        ELSE 0
    END                                                               AS view_to_purchase_rate,
    CASE
        WHEN COUNT(*) FILTER (WHERE event_id = '{VIEW_EVENT}') > 0
        THEN COUNT(*) FILTER (WHERE event_id = '{LIKE_EVENT}') * 1.0
             / COUNT(*) FILTER (WHERE event_id = '{VIEW_EVENT}')
        ELSE 0
    END                                                               AS like_rate,
    MIN(stime)                                                         AS first_seen_ts,
    MAX(stime)                                                         AS last_seen_ts,
    DATEDIFF('day', MIN(DATE(stime)), MAX(DATE(stime)))               AS attention_lifespan_days
FROM merrec
GROUP BY item_id
""")

print("  item_interactions view created")
display(con.execute("SELECT * FROM item_interactions LIMIT 5").df())


In [ ]:
# ── 4b. Price features ────────────────────────────────────────────────────────
print("Computing item price features...")

con.execute("""
CREATE OR REPLACE VIEW item_price_tiered AS
WITH item_price AS (
    SELECT item_id, MEDIAN(price) AS median_price, LN(MEDIAN(price) + 1) AS log_price
    FROM merrec
    WHERE price IS NOT NULL AND price > 0
    GROUP BY item_id
)
SELECT item_id, median_price, log_price,
       NTILE(5) OVER (ORDER BY median_price) AS price_tier
FROM item_price
""")

print("  item_price_tiered view created")
display(con.execute("""
    SELECT price_tier, COUNT(*) AS n_items, ROUND(AVG(median_price), 2) AS avg_price
    FROM item_price_tiered GROUP BY price_tier ORDER BY price_tier
""").df())


In [ ]:
# ── 4c. Category context ──────────────────────────────────────────────────────
print("Computing item category features (mode category per item)...")

con.execute("""
CREATE OR REPLACE VIEW item_category AS
WITH cat_counts AS (
    SELECT item_id, c0_name, c1_name, c2_name, COUNT(*) AS cnt
    FROM merrec WHERE c0_name IS NOT NULL
    GROUP BY item_id, c0_name, c1_name, c2_name
)
SELECT DISTINCT ON (item_id)
    item_id, c0_name AS category_l0, c1_name AS category_l1, c2_name AS category_l2
FROM cat_counts
ORDER BY item_id, cnt DESC
""")

print("  item_category view created")


In [ ]:
# ── 4d. Write final item feature table ───────────────────────────────────────
COLD_START_K = 20
print(f"Writing item features (cold-start threshold k = {COLD_START_K})...")

con.execute(f"""
COPY (
    SELECT
        i.item_id,
        i.total_events,
        i.unique_users,
        i.view_count,
        i.like_count,
        i.cart_count,
        i.purchase_count,
        i.session_reach,
        LN(i.total_events + 1)                AS log_total_events,
        i.view_to_purchase_rate,
        i.like_rate,
        i.first_seen_ts,
        i.last_seen_ts,
        GREATEST(i.attention_lifespan_days, 1) AS attention_lifespan_days,
        p.median_price,
        p.log_price,
        p.price_tier,
        c.category_l0,
        c.category_l1,
        c.category_l2,
        CASE WHEN i.total_events <= {COLD_START_K} THEN 1 ELSE 0 END AS cold_start_flag,
        NTILE(4) OVER (ORDER BY i.total_events)  AS interaction_tier
    FROM item_interactions i
    LEFT JOIN item_price_tiered p USING(item_id)
    LEFT JOIN item_category     c USING(item_id)
    ORDER BY i.item_id
) TO '{ITEM_FEATURES_PATH}' (FORMAT PARQUET, COMPRESSION SNAPPY)
""")

n    = con.execute(f"SELECT COUNT(*) FROM read_parquet('{ITEM_FEATURES_PATH}')").fetchone()[0]
cols = con.execute(f"DESCRIBE SELECT * FROM read_parquet('{ITEM_FEATURES_PATH}')").df()
print(f"  item_features.parquet written")
print(f"  Rows: {n:,}   Columns: {len(cols)}")


---
## 5. Session Feature Table

One row per session. This is the direct operationalisation of the short-term intent signal — the counterpart to the long-term user features.

`high_intent_flag` is the most theoretically important column: a session containing a cart or purchase event is a strong short-term signal, and comparing its distribution against `user.purchase_count` directly addresses Research Question 2.

`max_event_weight` encodes the highest-intent event on a 0–3 ordinal scale: view (0), like (1), cart (2), purchase (3). This can be used as an implicit feedback weight in iALS/BPR-MF.


In [ ]:
# ── 5a. Core session aggregates ──────────────────────────────────────────────
print("Computing session core features...")

con.execute(f"""
CREATE OR REPLACE VIEW session_core AS
SELECT
    session_id,
    user_id,
    COUNT(*)                                                           AS session_length,
    COUNT(DISTINCT item_id)                                            AS unique_items_viewed,
    COUNT(DISTINCT event_id)                                           AS event_diversity,
    COUNT(DISTINCT c0_name)                                            AS unique_categories,
    MIN(stime)                                                          AS session_start,
    MAX(stime)                                                          AS session_end,
    DATEDIFF('second', MIN(stime), MAX(stime))                         AS session_duration_sec,
    COUNT(*) FILTER (WHERE event_id = '{VIEW_EVENT}')                 AS session_view_count,
    COUNT(*) FILTER (WHERE event_id = '{LIKE_EVENT}')                 AS session_like_count,
    COUNT(*) FILTER (WHERE event_id = '{CART_EVENT}')                 AS session_cart_count,
    COUNT(*) FILTER (WHERE event_id = '{PURCHASE_EVENT}')             AS session_purchase_count,
    CASE WHEN COUNT(*) = 1 THEN 1 ELSE 0 END                         AS bounce_flag,
    CASE WHEN COUNT(*) FILTER (WHERE event_id IN (
             '{CART_EVENT}', '{PURCHASE_EVENT}')) > 0
         THEN 1 ELSE 0 END                                             AS high_intent_flag,
    MAX(CASE
        WHEN event_id = '{PURCHASE_EVENT}' THEN 3
        WHEN event_id = '{CART_EVENT}'     THEN 2
        WHEN event_id = '{LIKE_EVENT}'     THEN 1
        ELSE 0
    END)                                                               AS max_event_weight
FROM merrec
GROUP BY session_id, user_id
""")

print("  session_core view created")
display(con.execute("SELECT * FROM session_core LIMIT 5").df())


In [ ]:
# ── 5b. Dominant category per session ────────────────────────────────────────
con.execute("""
CREATE OR REPLACE VIEW session_category AS
WITH cat_counts AS (
    SELECT session_id, c0_name, COUNT(*) AS cnt
    FROM merrec WHERE c0_name IS NOT NULL
    GROUP BY session_id, c0_name
)
SELECT DISTINCT ON (session_id) session_id, c0_name AS dominant_category_l0
FROM cat_counts ORDER BY session_id, cnt DESC
""")
print("  session_category view created")


In [ ]:
# ── 5c. Average inter-event gap (sampled last 30 days) ───────────────────────
#
# Measures browsing pace. Fast sessions (small gap) = purposeful search.
# Slow sessions (large gap) = passive browsing.
# Sampled to avoid a full 1B-row ordered window scan.

con.execute("""
CREATE OR REPLACE VIEW session_gap AS
WITH recent AS (SELECT MAX(stime) - INTERVAL '30 days' AS cutoff FROM merrec),
ordered AS (
    SELECT session_id, stime,
           LEAD(stime) OVER (PARTITION BY session_id ORDER BY stime) AS next_ts
    FROM merrec
    WHERE stime >= (SELECT cutoff FROM recent)
)
SELECT session_id,
       AVG(DATEDIFF('second', stime, next_ts)) AS avg_inter_event_gap_sec
FROM ordered
WHERE next_ts IS NOT NULL
GROUP BY session_id
""")
print("  session_gap view created (sampled last 30 days)")


In [ ]:
# ── 5d. Write final session feature table ────────────────────────────────────
print("Joining session views and writing to Parquet...")

con.execute(f"""
COPY (
    SELECT
        s.session_id,
        s.user_id,
        s.session_length,
        s.unique_items_viewed,
        s.event_diversity,
        s.unique_categories,
        s.session_start,
        s.session_end,
        s.session_duration_sec,
        s.session_view_count,
        s.session_like_count,
        s.session_cart_count,
        s.session_purchase_count,
        s.bounce_flag,
        s.high_intent_flag,
        s.max_event_weight,
        c.dominant_category_l0,
        g.avg_inter_event_gap_sec
    FROM session_core s
    LEFT JOIN session_category c USING(session_id)
    LEFT JOIN session_gap      g USING(session_id)
    ORDER BY s.session_id
) TO '{SESSION_FEATURES_PATH}' (FORMAT PARQUET, COMPRESSION SNAPPY)
""")

n    = con.execute(f"SELECT COUNT(*) FROM read_parquet('{SESSION_FEATURES_PATH}')").fetchone()[0]
cols = con.execute(f"DESCRIBE SELECT * FROM read_parquet('{SESSION_FEATURES_PATH}')").df()
print(f"  session_features.parquet written")
print(f"  Rows: {n:,}   Columns: {len(cols)}")


---
## 6. Feature Validation & Quality Checks

Four checks before these tables move to Layer 3:

1. **Null rate audit** — flag any column above 30% missing
2. **Distribution sanity** — key features visualised to catch encoding errors
3. **Cold-start coverage** — confirm k = 20 produces a meaningful split
4. **Cross-table user coverage** — confirm all users appear in session table


In [ ]:
# ── 6a. Null rate audit ───────────────────────────────────────────────────────
sns.set_theme(style="whitegrid", font_scale=1.0)

for label, path in [
    ("User features",    USER_FEATURES_PATH),
    ("Item features",    ITEM_FEATURES_PATH),
    ("Session features", SESSION_FEATURES_PATH),
]:
    all_cols = con.execute(f"DESCRIBE SELECT * FROM read_parquet('{path}')").df()["column_name"].tolist()

    select_nulls = ", ".join([
        f"AVG(CASE WHEN \"{c}\" IS NULL THEN 1.0 ELSE 0.0 END) AS \"{c}\""
        for c in all_cols
    ])
    null_df = con.execute(f"SELECT {select_nulls} FROM read_parquet('{path}')").df()
    null_rates = null_df.T.rename(columns={0: "null_rate"}).sort_values("null_rate", ascending=False)
    high_null  = null_rates[null_rates["null_rate"] > 0.01]

    fig, ax = plt.subplots(figsize=(9, max(3, len(high_null) * 0.4)))
    ax.barh(high_null.index[::-1], high_null["null_rate"][::-1] * 100,
            color="steelblue", edgecolor="none", alpha=0.8)
    ax.axvline(30, color="crimson", linestyle="--", linewidth=0.9, label="30% warning")
    ax.set(xlabel="Null rate (%)", title=f"{label} — Null Rate per Column (>1%)")
    ax.legend(fontsize=8)
    plt.tight_layout()
    plt.show()
    print(f"  {label}: {len(all_cols)} total columns, {len(high_null)} with >1% nulls\n")


In [ ]:
# ── 6b. Key feature distributions ────────────────────────────────────────────
fig, axes = plt.subplots(2, 3, figsize=(14, 8))
axes = axes.flatten()

uf = con.execute(f"SELECT log_frequency      FROM read_parquet('{USER_FEATURES_PATH}')").df()
ue = con.execute(f"SELECT category_entropy   FROM read_parquet('{USER_FEATURES_PATH}')").df()
ul = con.execute(f"SELECT active_lifespan_days FROM read_parquet('{USER_FEATURES_PATH}')").df()
il = con.execute(f"SELECT log_total_events   FROM read_parquet('{ITEM_FEATURES_PATH}')").df()
pt = con.execute(f"""SELECT price_tier, COUNT(*) AS n FROM read_parquet('{ITEM_FEATURES_PATH}')
                       WHERE price_tier IS NOT NULL GROUP BY price_tier ORDER BY price_tier""").df()
sl = con.execute(f"SELECT session_length     FROM read_parquet('{SESSION_FEATURES_PATH}')").df()

axes[0].hist(uf["log_frequency"].dropna(),      bins=60, color="#7F77DD", edgecolor="none", alpha=0.85)
axes[0].set(xlabel="log(frequency + 1)",       title="User: log frequency")

axes[1].hist(ue["category_entropy"].dropna(),   bins=60, color="#7F77DD", edgecolor="none", alpha=0.85)
axes[1].set(xlabel="Category entropy (bits)",  title="User: category entropy")

axes[2].hist(ul["active_lifespan_days"].dropna(), bins=60, color="#7F77DD", edgecolor="none", alpha=0.85)
axes[2].set(xlabel="Days",                      title="User: active lifespan")

axes[3].hist(il["log_total_events"].dropna(),   bins=60, color="#1D9E75", edgecolor="none", alpha=0.85)
axes[3].set(xlabel="log(total events + 1)",    title="Item: log total events")

axes[4].bar(pt["price_tier"].astype(str), pt["n"], color="#1D9E75", edgecolor="none", alpha=0.85)
axes[4].set(xlabel="Price tier (Q1=cheapest)", title="Item: price tier distribution")

axes[5].hist(sl["session_length"].clip(upper=100), bins=60, color="#D85A30", edgecolor="none", alpha=0.85)
axes[5].set(xlabel="Events (clipped at 100)",  title="Session: session length")

fig.suptitle("Key Feature Distributions — Sanity Check", fontsize=13, fontweight="bold")
plt.tight_layout()
plt.show()


In [ ]:
# ── 6c. Cold-start coverage ───────────────────────────────────────────────────
COLD_START_K = 20

cold = con.execute(f"""
SELECT
    SUM(cold_start_flag)                            AS cold_start_items,
    COUNT(*)                                        AS total_items,
    ROUND(SUM(cold_start_flag) * 100.0 / COUNT(*), 2) AS cold_start_pct,
    SUM(CASE WHEN cold_start_flag = 0 THEN 1 END)  AS warm_items
FROM read_parquet('{ITEM_FEATURES_PATH}')
""").df()

print(f"Cold-start threshold k = {COLD_START_K}")
display(cold)

fig, ax = plt.subplots(figsize=(5, 4))
ax.bar(
    [f"Cold-start\n(≤{COLD_START_K} events)", f"Warm\n(>{COLD_START_K} events)"],
    [cold["cold_start_items"].iloc[0], cold["warm_items"].iloc[0]],
    color=["#E24B4A", "#1D9E75"], edgecolor="none", alpha=0.85
)
ax.set(ylabel="Number of items", title=f"Item Cold-Start Split (k = {COLD_START_K})")
for bar in ax.patches:
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() * 1.01,
            f"{bar.get_height():,.0f}", ha="center", fontsize=9)
plt.tight_layout()
plt.show()


In [ ]:
# ── 6d. Cross-table user coverage ─────────────────────────────────────────────
n_uf = con.execute(f"SELECT COUNT(*) FROM read_parquet('{USER_FEATURES_PATH}')").fetchone()[0]
n_sf = con.execute(f"SELECT APPROX_COUNT_DISTINCT(user_id) FROM read_parquet('{SESSION_FEATURES_PATH}')").fetchone()[0]
coverage = n_sf / n_uf * 100 if n_uf > 0 else 0

print(f"Users in user_features              : {n_uf:,}")
print(f"Distinct users in session_features  : {n_sf:,}")
print(f"Session coverage                    : {coverage:.1f}%")


---
## 7. Cross-Table Join Sample

A preview of the combined row that Layer 3 will use. User features (long-term signal) and session features (short-term signal) are joined via `user_id` and `session_id`; item features join on `item_id` from a sample of raw events.

This is the core structure of Research Question 2: every row has both a long-term user profile and a short-term session context for the same interaction, enabling direct comparison of their predictive contribution.


In [ ]:
sample_join = con.execute(f"""
WITH sample_sessions AS (
    SELECT DISTINCT session_id
    FROM read_parquet('{SESSION_FEATURES_PATH}')
    LIMIT 1000
),
sample_events AS (
    SELECT m.session_id, m.user_id, m.item_id
    FROM merrec m
    JOIN sample_sessions s USING(session_id)
    QUALIFY ROW_NUMBER() OVER (PARTITION BY m.session_id ORDER BY m.stime) = 1
)
SELECT
    e.session_id, e.user_id, e.item_id,

    -- Long-term user signal
    u.recency_days, u.frequency, u.purchase_count,
    u.category_entropy, u.active_lifespan_days,
    u.session_count, u.recency_tier, u.frequency_tier,

    -- Short-term session signal
    s.session_length, s.high_intent_flag, s.bounce_flag,
    s.max_event_weight, s.dominant_category_l0,

    -- Item signal
    i.total_events     AS item_total_events,
    i.cold_start_flag,
    i.price_tier,
    i.category_l0      AS item_category_l0,
    i.view_to_purchase_rate AS item_conversion_rate

FROM sample_events e
LEFT JOIN read_parquet('{USER_FEATURES_PATH}')    u ON e.user_id    = u.user_id
LEFT JOIN read_parquet('{SESSION_FEATURES_PATH}') s ON e.session_id = s.session_id
LEFT JOIN read_parquet('{ITEM_FEATURES_PATH}')    i ON e.item_id    = i.item_id
""").df()

print(f"Join sample shape: {sample_join.shape}")
display(sample_join.head(10))
print("\nDescriptive statistics (numeric columns):")
display(sample_join.describe().round(3))


---
## Summary

Layer 2 has produced three feature tables saved to `DATASET_ROOT/features/`:

| File | Rows | Key features |
|------|------|-------------|
| `user_features.parquet` | one per user | recency, frequency, purchase count, category entropy, active lifespan, session count, RFM tiers |
| `item_features.parquet` | one per item | total events, conversion rate, price tier, category hierarchy, cold_start_flag, interaction tier |
| `session_features.parquet` | one per session | session length, duration, bounce flag, high_intent_flag, max_event_weight, dominant category |

### Methodology notes for your report

- Cold-start threshold **k = 20**: justified by the long-tail analysis in Layer 1; items below this threshold have insufficient interaction history for reliable embedding.
- `max_event_weight` (0–3 ordinal) can be used directly as an implicit feedback confidence weight in iALS and BPR-MF, replacing the binary 0/1 signal used in baseline models.
- `avg_inter_event_gap_sec` is approximate (sampled last 30 days) — state this limitation explicitly in your methodology section.
- `recency_tier`, `frequency_tier`, `monetary_tier` are NTILE convenience columns — they are not independent features and should not be used alongside their continuous counterparts in the same model.

### Next: Layer 3 — Unsupervised Pattern Detection

These three tables feed directly into:
- **WK-means / DBSCAN** on user feature vectors → behavioral segments
- **iALS / BPR-MF** on user × item interactions weighted by `max_event_weight` → latent embeddings
- **HMM / change-point detection** on session sequences ordered by `session_start` → behavioral regime transitions


In [ ]:
con.close()
print("DuckDB connection closed.")
print("Layer 2 complete — all three feature tables written to DATASET_ROOT/features/")
